In [5]:
import pandas as pd
from typing import Dict, Tuple, List


datas = []

columns = ['model', 'dataset', 'recall', 'query_latency', 'construction_time']

# First CSV
df = pd.read_csv("graph_standalone_results_index_v1.csv")

for index, row in df.iterrows():
    data = [
        row['model'],
        row['dataset'],
        row['recall with 2k refinement'],
        1/row['qps with 2k refienement'],
        row['construction time']
    ]
    datas.append(data)

# Second CSV
df = pd.read_csv("graph_augmented_results_index_v1.csv")

for index, row in df.iterrows():
    data = [
        row['model1'] + '+' + row['model2'],
        row['dataset'],
        row["augmented model's recall with refinement"],
        1/row["augmented model's qps with refinement in parallal settings"],
        row["augmented model's construction time in parallal settings"]
    ]
    datas.append(data)

# Convert to DataFrame
final_df = pd.DataFrame(datas, columns=columns)

# Save to CSV
final_df.to_csv("combined_graph_results.csv", index=False)

profiles_per_dataset: Dict[str, List[Tuple[float, float]]] = {}

for dataset, group in final_df.groupby("dataset"):
    g = group.copy()
    g = g[pd.notna(g["query_latency"]) & (g["query_latency"] > 0)]
    g = g[pd.notna(g["construction_time"]) & (g["construction_time"] > 0)]
    latencies = g["query_latency"].astype(float)
    ctimes = g["construction_time"].astype(float)
    lat_q = latencies.quantile([0.25, 0.5]).tolist()
    cst_q = ctimes.quantile([0.25, 0.5]).tolist()
    profiles: List[Tuple[float, float, float]] = []
    for (Tq, Tc) in zip(lat_q, cst_q):
        profiles.append((float(Tq), float(Tc)))

    profiles_per_dataset[dataset] = profiles
    
_500k_recalls=[]
_100k_recalls=[]
_50k_recalls=[]
_10k_recalls=[] 
recalls=[] 
    
for dataset in profiles_per_dataset.keys():
    base_best_recall=0
    augmented_best_recall=0
    dataset_filtered_df=final_df[final_df['dataset']==dataset]
    for index, row in dataset_filtered_df.iterrows():
        if '+' in row['model'] and row['query_latency']<=profiles_per_dataset[dataset][0][0] and row['construction_time']<=profiles_per_dataset[dataset][0][1]:
            augmented_best_recall=max(augmented_best_recall,row['recall'])
        elif (not '+' in row['model']) and row['query_latency']<=profiles_per_dataset[dataset][0][0] and row['construction_time']<=profiles_per_dataset[dataset][0][1]:
            base_best_recall=max(base_best_recall,row['recall'])
    # print(dataset, base_best_recall, augmented_best_recall)
    if '500k' in dataset:
        _500k_recalls.append((base_best_recall,augmented_best_recall))
    elif '100k' in dataset:
        _100k_recalls.append((base_best_recall,augmented_best_recall))
    elif '50k' in dataset:
        _50k_recalls.append((base_best_recall,augmented_best_recall))
    elif '10k' in dataset:
        _10k_recalls.append((base_best_recall,augmented_best_recall))
    else:
        recalls.append((base_best_recall,augmented_best_recall))
        
    
    
print("Hard Constraint")

for label, data in {
    "500k": _500k_recalls,
    "100k": _100k_recalls,
    "50k": _50k_recalls,
    "10k": _10k_recalls,
    "1M": recalls
}.items():
    if data:
        base = sum(x[0] for x in data)/len(data)
        aug  = sum(x[1] for x in data)/len(data)
        print(label, base, aug)

_500k_recalls=[]
_100k_recalls=[]
_50k_recalls=[]
_10k_recalls=[] 
recalls=[] 

for dataset in profiles_per_dataset.keys():
    base_best_recall=0
    augmented_best_recall=0
    dataset_filtered_df=final_df[final_df['dataset']==dataset]
    for index, row in dataset_filtered_df.iterrows():
        if '+' in row['model'] and row['query_latency']<=profiles_per_dataset[dataset][1][0] and row['construction_time']<=profiles_per_dataset[dataset][1][1]:
            augmented_best_recall=max(augmented_best_recall,row['recall'])
        elif (not '+' in row['model']) and row['query_latency']<=profiles_per_dataset[dataset][1][0] and row['construction_time']<=profiles_per_dataset[dataset][1][1]:
            base_best_recall=max(base_best_recall,row['recall'])
    # print(dataset, base_best_recall, augmented_best_recall)
    if '500k' in dataset:
        _500k_recalls.append((base_best_recall,augmented_best_recall))
    if '100k' in dataset:
        _100k_recalls.append((base_best_recall,augmented_best_recall))
    if '50k' in dataset:
        _50k_recalls.append((base_best_recall,augmented_best_recall))
    if '10k' in dataset:
        _10k_recalls.append((base_best_recall,augmented_best_recall))
    else:
        recalls.append((base_best_recall,augmented_best_recall))


print("Relaxed Constraint")

for label, data in {
    "500k": _500k_recalls,
    "100k": _100k_recalls,
    "50k": _50k_recalls,
    "10k": _10k_recalls,
    "1M": recalls
}.items():
    if data:
        base = sum(x[0] for x in data)/len(data)
        aug  = sum(x[1] for x in data)/len(data)
        print(label, base, aug)

Hard Constraint
500k 0.8032755714285714 0.8958302857142856
100k 0.888942 0.9341948571428571
50k 0.8931014285714285 0.9479538571428571
10k 0.9731291428571429 0.9845334285714287
1M 0.7860228571428571 0.8950021428571429
Relaxed Constraint
500k 0.9271255714285714 0.9540785714285713
100k 0.9443302857142858 0.9656649999999999
50k 0.9495408571428572 0.9691428571428572
10k 0.9807345714285713 0.9913905714285713
1M 0.9327961428571433 0.9572993928571429


In [6]:
import pandas as pd
from typing import Dict, Tuple, List

datas = []
        

columns = ['model', 'dataset', 'recall', 'query_latency', 'construction_time']

# First CSV
df = pd.read_csv("graph_standalone_results_rc_v2.csv")

for index, row in df.iterrows():
    data = [
        row['model'],
        row['dataset'],
        row['recall with 2k refinement'],
        1/row['qps with 2k refienement'],
        row['construction time']
    ]
    datas.append(data)

# Second CSV
df = pd.read_csv("graph_augmented_results_rc_v2.csv")

for index, row in df.iterrows():
    data = [
        row['model1'] + '+' + row['model2'],
        row['dataset'],
        row["augmented model's recall with refinement"],
        1/row["augmented model's qps with refinement in parallal settings"],
        row["augmented model's construction time in parallal settings"]
    ]
    datas.append(data)

# Convert to DataFrame
final_df = pd.DataFrame(datas, columns=columns)

# Save to CSV
final_df.to_csv("combined_graph_results_rc.csv", index=False)

profiles_per_dataset: Dict[str, List[Tuple[float, float]]] = {}

for dataset, group in final_df.groupby("dataset"):
    g = group.copy()
    g = g[pd.notna(g["query_latency"]) & (g["query_latency"] > 0)]
    g = g[pd.notna(g["construction_time"]) & (g["construction_time"] > 0)]
    latencies = g["query_latency"].astype(float)
    ctimes = g["construction_time"].astype(float)
    lat_q = latencies.quantile([0.25, 0.5]).tolist()
    cst_q = ctimes.quantile([0.25, 0.5]).tolist()
    profiles: List[Tuple[float, float, float]] = []
    for (Tq, Tc) in zip(lat_q, cst_q):
        profiles.append((float(Tq), float(Tc)))
    # if is_valid(dataset):
    profiles_per_dataset[dataset] = profiles
    
_500k_recalls=[]
_100k_recalls=[]
_50k_recalls=[]
_10k_recalls=[] 
# print(profiles_per_dataset.keys())
for dataset in profiles_per_dataset.keys():
    base_best_recall=0
    augmented_best_recall=0
    dataset_filtered_df=final_df[final_df['dataset']==dataset]
    for index, row in dataset_filtered_df.iterrows():
        if '+' in row['model'] and row['query_latency']<=profiles_per_dataset[dataset][0][0] and row['construction_time']<=profiles_per_dataset[dataset][0][1]:
            augmented_best_recall=max(augmented_best_recall,row['recall'])
        elif (not '+' in row['model']) and row['query_latency']<=profiles_per_dataset[dataset][0][0] and row['construction_time']<=profiles_per_dataset[dataset][0][1]:
            base_best_recall=max(base_best_recall,row['recall'])
    # print(dataset, base_best_recall, augmented_best_recall)
#     print(dataset, base_best_recall, augmented_best_recall)
    if '_rc_1.25' in dataset:
        _500k_recalls.append((base_best_recall,augmented_best_recall))
    if '_rc_1.5' in dataset:
        _100k_recalls.append((base_best_recall,augmented_best_recall))
    if '_rc_2.0' in dataset:
        _50k_recalls.append((base_best_recall,augmented_best_recall))
    
    
print("Hard Constraint")

for label, data in {
    "_rc_1.25": _500k_recalls,
    "_rc_1.5": _100k_recalls,
    "_rc_2.0": _50k_recalls
}.items():
    if data:
        base = sum(x[0] for x in data)/len(data)
        aug  = sum(x[1] for x in data)/len(data)
        print(label, base, aug)


_500k_recalls=[]
_100k_recalls=[]
_50k_recalls=[]
_10k_recalls=[] 

for dataset in profiles_per_dataset.keys():
    base_best_recall=0
    augmented_best_recall=0
    dataset_filtered_df=final_df[final_df['dataset']==dataset]
    for index, row in dataset_filtered_df.iterrows():
        if '+' in row['model'] and row['query_latency']<=profiles_per_dataset[dataset][1][0] and row['construction_time']<=profiles_per_dataset[dataset][1][1]:
            augmented_best_recall=max(augmented_best_recall,row['recall'])
        elif (not '+' in row['model']) and row['query_latency']<=profiles_per_dataset[dataset][1][0] and row['construction_time']<=profiles_per_dataset[dataset][1][1]:
            base_best_recall=max(base_best_recall,row['recall'])
    if '_rc_1.25' in dataset:
        _500k_recalls.append((base_best_recall,augmented_best_recall))
    if '_rc_1.5' in dataset:
        _100k_recalls.append((base_best_recall,augmented_best_recall))
    if '_rc_2.0' in dataset:
        _50k_recalls.append((base_best_recall,augmented_best_recall))


print("Relaxed Constraint")

for label, data in {
    "_rc_1.25": _500k_recalls,
    "_rc_1.5": _100k_recalls,
    "_rc_2.0": _50k_recalls
}.items():
    if data:
        base = sum(x[0] for x in data)/len(data)
        aug  = sum(x[1] for x in data)/len(data)
        print(label, base, aug)
        
        


Hard Constraint
_rc_1.25 0.5528235294117647 0.6305588235294116
_rc_1.5 0.687735294117647 0.7616470588235295
_rc_2.0 0.7949411764705883 0.8597941176470587
Relaxed Constraint
_rc_1.25 0.6212941176470588 0.70125
_rc_1.5 0.7519999999999999 0.8094558823529412
_rc_2.0 0.8500147058823527 0.8943970588235294


In [9]:
import pandas as pd
from typing import Dict, Tuple, List



all_datasets=[
    'MNIST',
    'coco-nomic-768-normalized_20_variants',
    'agnews-mxbai-1024-euclidean_20_variants',
    'gooaq-distilroberta-768-normalized_20_variants',
    'ccnews-nomic-768-normalized_20_variants',
    'arxiv-nomic-768-normalized_20_variants'
]

datas = []

def is_valid(dataset_name):
    for _dataset in all_datasets:
        if _dataset in dataset_name:
            return True
    return False
        

columns = ['model', 'dataset', 'recall', 'query_latency', 'construction_time']

# First CSV
df = pd.read_csv("graph_standalone_results_dimension_analysis.csv")

for index, row in df.iterrows():
    data = [
        row['model'],
        row['dataset'],
        row['recall with 2k refinement'],
        1/row['qps with 2k refienement'],
        row['construction time']
    ]
    datas.append(data)

# Second CSV
df = pd.read_csv("graph_augmented_results_dimension_analysis.csv")

for index, row in df.iterrows():
    data = [
        row['model1'] + '+' + row['model2'],
        row['dataset'],
        row["augmented model's recall with refinement"],
        1/row["augmented model's qps with refinement in parallal settings"],
        row["augmented model's construction time in parallal settings"]
    ]
    datas.append(data)

# Convert to DataFrame
final_df = pd.DataFrame(datas, columns=columns)

# Save to CSV
final_df.to_csv("combined_graph_results_dimension.csv", index=False)

profiles_per_dataset: Dict[str, List[Tuple[float, float]]] = {}

for dataset, group in final_df.groupby("dataset"):
    g = group.copy()
    g = g[pd.notna(g["query_latency"]) & (g["query_latency"] > 0)]
    g = g[pd.notna(g["construction_time"]) & (g["construction_time"] > 0)]
    latencies = g["query_latency"].astype(float)
    ctimes = g["construction_time"].astype(float)
    lat_q = latencies.quantile([0.25, 0.5]).tolist()
    cst_q = ctimes.quantile([0.25, 0.5]).tolist()
    profiles: List[Tuple[float, float, float]] = []
    for (Tq, Tc) in zip(lat_q, cst_q):
        profiles.append((float(Tq), float(Tc)))
    if is_valid(dataset):
        profiles_per_dataset[dataset] = profiles
    
_500k_recalls=[]
_100k_recalls=[]
_50k_recalls=[]
_10k_recalls=[] 
# print(profiles_per_dataset.keys())
for dataset in profiles_per_dataset.keys():
    base_best_recall=0
    augmented_best_recall=0
    dataset_filtered_df=final_df[final_df['dataset']==dataset]
    for index, row in dataset_filtered_df.iterrows():
        if '+' in row['model'] and row['query_latency']<=profiles_per_dataset[dataset][0][0] and row['construction_time']<=profiles_per_dataset[dataset][0][1]:
            augmented_best_recall=max(augmented_best_recall,row['recall'])
        elif (not '+' in row['model']) and row['query_latency']<=profiles_per_dataset[dataset][0][0] and row['construction_time']<=profiles_per_dataset[dataset][0][1]:
            base_best_recall=max(base_best_recall,row['recall'])
    # print(dataset, base_best_recall, augmented_best_recall)
#     print(dataset, base_best_recall, augmented_best_recall)
    if '768_dim' in dataset:
        _500k_recalls.append((base_best_recall,augmented_best_recall))
    if '512_dim' in dataset:
        _100k_recalls.append((base_best_recall,augmented_best_recall))
    if '256_dim' in dataset:
        _50k_recalls.append((base_best_recall,augmented_best_recall))
    if '128_dim' in dataset:
        _10k_recalls.append((base_best_recall,augmented_best_recall))
    
    
print("Hard Constraint")

for label, data in {
    "768_dim": _500k_recalls,
    "512_dim": _100k_recalls,
    "256_dim": _50k_recalls,
    "128_dim": _10k_recalls
}.items():
    if data:
        base = sum(x[0] for x in data)/len(data)
        aug  = sum(x[1] for x in data)/len(data)
        print(label, base, aug)


_500k_recalls=[]
_100k_recalls=[]
_50k_recalls=[]
_10k_recalls=[] 

for dataset in profiles_per_dataset.keys():
    base_best_recall=0
    augmented_best_recall=0
    dataset_filtered_df=final_df[final_df['dataset']==dataset]
    for index, row in dataset_filtered_df.iterrows():
        if '+' in row['model'] and row['query_latency']<=profiles_per_dataset[dataset][1][0] and row['construction_time']<=profiles_per_dataset[dataset][1][1]:
            augmented_best_recall=max(augmented_best_recall,row['recall'])
        elif (not '+' in row['model']) and row['query_latency']<=profiles_per_dataset[dataset][1][0] and row['construction_time']<=profiles_per_dataset[dataset][1][1]:
            base_best_recall=max(base_best_recall,row['recall'])
    # print(dataset, base_best_recall, augmented_best_recall)
    if '768_dim' in dataset:
        _500k_recalls.append((base_best_recall,augmented_best_recall))
    if '512_dim' in dataset:
        _100k_recalls.append((base_best_recall,augmented_best_recall))
    if '256_dim' in dataset:
        _50k_recalls.append((base_best_recall,augmented_best_recall))
    if '128_dim' in dataset:
        _10k_recalls.append((base_best_recall,augmented_best_recall))


print("Relaxed Constraint")

for label, data in {
    "768_dim": _500k_recalls,
    "512_dim": _100k_recalls,
    "256_dim": _50k_recalls,
    "128_dim": _10k_recalls
}.items():
    if data:
        base = sum(x[0] for x in data)/len(data)
        aug  = sum(x[1] for x in data)/len(data)
        print(label, base, aug)

Hard Constraint
768_dim 0.9187916666666668 0.9457916666666666
512_dim 0.8522083333333331 0.8785416666666667
256_dim 0.7827083333333333 0.8187083333333334
128_dim 0.7820416666666666 0.8177083333333334
Relaxed Constraint
768_dim 0.9478749999999999 0.9682083333333332
512_dim 0.879375 0.9051666666666666
256_dim 0.8162083333333333 0.861125
128_dim 0.8203333333333332 0.8445833333333334
